# Flask RAG Chatbot (Gemini,Streaming)

A domain-specific RAG chatbot grounded in the **Flask** source code (github.com/pallets/flask).

Differences from a plain-text-document RAG pipeline:
- **AST-based chunking** — splits code by function/class/method boundaries instead of blind character windows, so retrieval doesn't return a chunk cut off mid-function
- **Code-aware metadata** — every chunk remembers its file path and qualified name (e.g. `app.py :: Flask.add_url_rule`), so answers can cite exactly where something lives
- **Streaming generation** — responses print token-by-token as Gemini generates them
- **Code-preserving cleanup** — markdown stripping leaves fenced code blocks untouched, since code formatting is the actual content here, not decoration

## Step 1: Install dependencies

## Step 2: Load the Gemini API key

In [ ]:
import os
import time
from google.colab import userdata
from google import genai
from google.genai import errors as genai_errors

api_key = os.dotenv.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

CHAT_MODEL = "gemini-flash-latest"      # alias -> current recommended Flash model
EMBED_MODEL = "gemini-embedding-001"    # GA text embedding model

print("Client ready.")

In [31]:
def generate_with_retry(model, contents, max_retries=8, config=None):
    attempt = 0
    while True:
        try:
            kwargs = {"model": model, "contents": contents}
            if config is not None:
                kwargs["config"] = config
            return client.models.generate_content(**kwargs)
        except genai_errors.ClientError as e:
            if "429" in str(e):
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 60)
                print(f"Rate limited on generate_content - waiting {wait}s (attempt {attempt}/{max_retries})...")
                time.sleep(wait)
            else:
                raise


def stream_with_retry(model, contents, max_retries=8, config=None):
    attempt = 0
    while True:
        try:
            kwargs = {"model": model, "contents": contents}
            if config is not None:
                kwargs["config"] = config
            for chunk in client.models.generate_content_stream(**kwargs):
                yield chunk
            return
        except genai_errors.ClientError as e:
            if "429" in str(e):
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 60)
                print(f"Rate limited on stream - waiting {wait}s (attempt {attempt}/{max_retries})...")
                time.sleep(wait)
            else:
                raise

## Step 3: Download the Flask repository

No `git clone` needed in Colab — just pull the zip archive of the main branch directly.

In [19]:

import os

REPO_ZIP_URL = "https://github.com/pallets/flask/archive/refs/heads/main.zip"
REPO_DIR = "repo"

print("Downloading Flask repository...")
resp = requests.get(REPO_ZIP_URL)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall(REPO_DIR)

# The zip extracts into a single top-level folder like "flask-main" - find it
extracted_root = os.path.join(REPO_DIR, os.listdir(REPO_DIR)[0])
print(f"Extracted to: {extracted_root}")

Extracted to: repo/flask-main


## Step 4: Collect Python source files

Excludes tests, docs, and CI config — keeps the actual library source.

In [20]:
EXCLUDE_DIRS = {"tests", "test", "docs", ".github", "examples", ".git"}

py_files = []
for root, dirs, files in os.walk(extracted_root):
    dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
    for fname in files:
        if fname.endswith(".py"):
            py_files.append(os.path.join(root, fname))

print(f"Found {len(py_files)} Python files.")
for f in py_files[:10]:
    print(" ", os.path.relpath(f, extracted_root))
if len(py_files) > 10:
    print(f"  ... and {len(py_files) - 10} more")

Found 24 Python files.
  src/flask/globals.py
  src/flask/ctx.py
  src/flask/config.py
  src/flask/sessions.py
  src/flask/__init__.py
  src/flask/signals.py
  src/flask/helpers.py
  src/flask/wrappers.py
  src/flask/views.py
  src/flask/app.py
  ... and 14 more


## Step 5: AST-based chunking

Splits each file by class and function/method boundaries rather than fixed character windows — a chunk boundary lands on a code boundary, not mid-function.

In [21]:
import ast

def extract_chunks_from_source(filepath, source, rel_path):
    """Walks the AST and produces one chunk per top-level function, per method,
    and a summary chunk per class (signature + docstring + method list).
    Nested/inner functions are intentionally not split out separately, to keep
    chunks at a granularity useful for retrieval rather than over-fragmenting."""
    chunks = []

    try:
        tree = ast.parse(source)
    except SyntaxError:
        return chunks  # skip files that don't parse (rare, e.g. Python 2 syntax)

    def visit(node, class_name=None):
        for child in ast.iter_child_nodes(node):
            if isinstance(child, ast.ClassDef):
                methods = [n.name for n in child.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
                docstring = ast.get_docstring(child) or ""
                overview = f"class {child.name}:\n\"\"\"{docstring}\"\"\"\nMethods: {', '.join(methods)}"
                chunks.append({
                    "text": overview,
                    "name": child.name,
                    "type": "class",
                    "file": rel_path,
                    "lineno": child.lineno,
                })
                visit(child, class_name=child.name)  # descend into methods

            elif isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                segment = ast.get_source_segment(source, child)
                if segment is None:
                    continue
                qualified_name = f"{class_name}.{child.name}" if class_name else child.name
                chunks.append({
                    "text": segment,
                    "name": qualified_name,
                    "type": "method" if class_name else "function",
                    "file": rel_path,
                    "lineno": child.lineno,
                })
                # do not descend further - keeps chunk granularity at function level

            else:
                visit(child, class_name)

    visit(tree)

    # Also capture the module-level docstring + imports as one small chunk (useful context)
    module_docstring = ast.get_docstring(tree)
    if module_docstring:
        chunks.append({
            "text": f"Module docstring for {rel_path}:\n{module_docstring}",
            "name": "<module>",
            "type": "module_docstring",
            "file": rel_path,
            "lineno": 1,
        })

    return chunks


all_chunks = []
for filepath in py_files:
    rel_path = os.path.relpath(filepath, extracted_root)
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        source = f.read()
    all_chunks.extend(extract_chunks_from_source(filepath, source, rel_path))

print(f"Total chunks extracted: {len(all_chunks)}")
print(f"\nExample chunk:")
print(f"  file: {all_chunks[0]['file']}")
print(f"  name: {all_chunks[0]['name']}")
print(f"  type: {all_chunks[0]['type']}")
print(f"  text:\n{all_chunks[0]['text'][:300]}")

Total chunks extracted: 411

Example chunk:
  file: src/flask/globals.py
  name: ProxyMixin
  type: class
  text:
class ProxyMixin:
""""""
Methods: _get_current_object


## Step 6: Embed chunks and build the vector store

In [ ]:
import chromadb
from google.genai import types
from google.genai import errors as genai_errors
import time

chroma_client = chromadb.PersistentClient(path="/content/chroma_db")

try:
    chroma_client.delete_collection("flask_code")
except Exception:
    pass
collection = chroma_client.create_collection("flask_code")

def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT", batch_size=20, max_retries=6):
    """Embeds texts in small, paced batches with retry-with-backoff on 429s -
    the free tier's embedding quota is easy to burst past with large batch sizes."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        attempt = 0
        while True:
            try:
                result = client.models.embed_content(
                    model=EMBED_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(task_type=task_type)
                )
                break
            except genai_errors.ClientError as e:
                if "429" in str(e):
                    attempt += 1
                    if attempt > max_retries:
                        raise
                    wait = min(2 ** attempt, 30)
                    print(f"Rate limited - waiting {wait}s before retry (attempt {attempt}/{max_retries})...")
                    time.sleep(wait)
                else:
                    raise

        all_embeddings.extend([e.values for e in result.embeddings])
        print(f"Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks...")
        time.sleep(1)  # pacing cushion to stay under the per-minute quota

    return all_embeddings

# Embed the "text" field of every chunk, prefixed with its qualified name for extra retrieval signal
texts_to_embed = [f"{c['name']} ({c['file']}):\n{c['text']}" for c in all_chunks]

print("Embedding chunks (this will take a few minutes for the full repo, paced to respect free-tier rate limits)...")
chunk_embeddings = embed_texts(texts_to_embed)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    embeddings=chunk_embeddings,
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"file": c["file"], "name": c["name"], "type": c["type"], "lineno": c["lineno"]} for c in all_chunks],
)

print(f"Indexed {len(all_chunks)} chunks into the vector store.")

Embedding chunks (this will take a few minutes for the full repo, paced to respect free-tier rate limits)...
Embedded 20/411 chunks...
Embedded 40/411 chunks...
Embedded 60/411 chunks...
Embedded 80/411 chunks...
Embedded 100/411 chunks...
Rate limited - waiting 2s before retry (attempt 1/6)...
Rate limited - waiting 4s before retry (attempt 2/6)...
Rate limited - waiting 8s before retry (attempt 3/6)...
Rate limited - waiting 16s before retry (attempt 4/6)...
Rate limited - waiting 30s before retry (attempt 5/6)...
Embedded 120/411 chunks...
Embedded 140/411 chunks...
Embedded 160/411 chunks...
Embedded 180/411 chunks...
Embedded 200/411 chunks...
Rate limited - waiting 2s before retry (attempt 1/6)...
Rate limited - waiting 4s before retry (attempt 2/6)...
Rate limited - waiting 8s before retry (attempt 3/6)...
Rate limited - waiting 16s before retry (attempt 4/6)...
Rate limited - waiting 30s before retry (attempt 5/6)...
Embedded 220/411 chunks...
Embedded 240/411 chunks...
Embedde

## Step 7: Retrieval function

In [ ]:
def retrieve(query, top_k=5):
    query_embedding = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)

    retrieved = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        retrieved.append({
            "text": doc,
            "file": meta["file"],
            "name": meta["name"],
            "type": meta["type"],
            "distance": dist,
        })
    return retrieved

# Quick test
test_results = retrieve("How does Flask handle URL routing?", top_k=5)
for r in test_results:
    print(f"[{r['file']} :: {r['name']}] ({r['type']}, distance={r['distance']:.3f})")

Embedded 1/1 chunks...
[src/flask/ctx.py :: AppContext.match_request] (method, distance=0.494)
[src/flask/app.py :: Flask.dispatch_request] (method, distance=0.502)
[src/flask/sansio/scaffold.py :: Scaffold.add_url_rule] (method, distance=0.564)
[src/flask/app.py :: Flask.url_for] (method, distance=0.566)
[src/flask/views.py :: View.dispatch_request] (method, distance=0.566)


## Step 8: Code-aware cleanup

Same markdown/LaTeX stripping as before, but fenced code blocks (```) are left completely untouched, since the code formatting inside them is the actual content, not decoration.

In [33]:
import re

def _strip_markdown_core(text):
    """Markdown/LaTeX stripping for plain prose, without a final strip() - boundary
    whitespace matters when this is rejoined with preserved code spans."""
    prev = None
    while prev != text:
        prev = text
        text = re.sub(r"\\text\{([^{}]*)\}", r"\1", text)

    text = re.sub(r"\\boxed\{([^{}]*)\}", r"\1", text)

    prev = None
    while prev != text:
        prev = text
        text = re.sub(r"\\frac\{([^{}]*)\}\{([^{}]*)\}", r"\1/\2", text)

    text = re.sub(r"\\times|\\cdot", "x", text)
    text = re.sub(r"\\(,|;|:|!|quad|qquad)", " ", text)
    text = re.sub(r"\\\[|\\\]|\\\(|\\\)", "", text)
    text = re.sub(r"\\[a-zA-Z]+", "", text)
    text = re.sub(r"[{}]", "", text)

    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"(\*\*)(.*?)\1", r"\2", text)  # bold: ** only
    text = re.sub(r"(?<!\w)\*(?!\s)(.*?)(?<!\s)\*(?!\w)", r"\1", text)  # italic: single * only, word-boundary guarded
    text = re.sub(r"~~(.*?)~~", r"\1", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = re.sub(r"^\s*[-*+]\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*\d+\.\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"^>\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*([-*_]\s*){3,}\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def strip_markdown_text(text):
    """Standalone version (e.g. for cleaning a full response with no code spans to preserve)."""
    return _strip_markdown_core(text).strip()


def clean_response(text):
    """Preserves fenced code blocks AND inline code spans untouched; only cleans
    markdown/LaTeX from plain prose outside of any code span. Boundary whitespace
    around preserved spans is kept intact by not stripping individual fragments."""
    fence_parts = re.split(r"(```.*?```)", text, flags=re.DOTALL)
    output = []
    for part in fence_parts:
        if part.startswith("```"):
            output.append(part)  # leave fenced code block exactly as-is
            continue
        inline_parts = re.split(r"(`[^`]*`)", part)
        for sub in inline_parts:
            if sub.startswith("`") and sub.endswith("`") and len(sub) >= 2:
                output.append(sub)  # leave inline code exactly as-is
            else:
                output.append(_strip_markdown_core(sub))
    return "".join(output).strip()

## Step 9: RAG generation with streaming

Prints tokens live as Gemini generates them, then shows a cleaned version once the full response is complete (code blocks preserved, prose cleaned).

In [34]:
RAG_SYSTEM_PROMPT = """You are a code assistant answering questions about the Flask web framework's source code.

Rules:
- Answer using ONLY the provided code/context snippets below.
- If the context doesn't contain enough information to answer, say so clearly rather than guessing.
- Do not use outside knowledge of Flask beyond what's in the provided context.
- Cite the file path and function/class name for any code you reference (e.g. "in app.py, Flask.add_url_rule").
- When showing code, use proper Markdown code fences with the python language tag.
- Be precise and concise.
"""

def rag_generate_stream(query, top_k=5, verbose=False):
    retrieved = retrieve(query, top_k=top_k)

    context_block = "\n\n".join(
        f"[{r['file']} :: {r['name']}]\n{r['text']}" for r in retrieved
    )

    full_prompt = f"""{RAG_SYSTEM_PROMPT}

Context:
{context_block}

Question: {query}

Answer:"""

    if verbose:
        print("--- Retrieved chunks ---")
        for r in retrieved:
            print(f"[{r['file']} :: {r['name']}] distance={r['distance']:.3f}")
        print()

    print("--- streaming ---")
    full_text = ""
    for chunk in client.models.generate_content_stream(model=CHAT_MODEL, contents=full_prompt):
        if chunk.text:
            print(chunk.text, end="", flush=True)
            full_text += chunk.text
    print()

    cleaned = clean_response(full_text)
    print("\n--- clean version ---")
    print(cleaned)
    return cleaned


_ = rag_generate_stream("How does Flask handle URL routing? Show the relevant code.", verbose=True)

Embedded 1/1 chunks...
--- Retrieved chunks ---
[src/flask/ctx.py :: AppContext.match_request] distance=0.423
[src/flask/app.py :: Flask.dispatch_request] distance=0.458
[src/flask/app.py :: Flask.create_url_adapter] distance=0.503
[src/flask/app.py :: Flask.url_for] distance=0.517
[src/flask/views.py :: View.dispatch_request] distance=0.523

--- streaming ---
Flask handles URL routing through a process of creating a URL adapter, matching incoming requests to routes, and dispatching the request to the matching view function:

### 1. Creating the URL Adapter
In `src/flask/app.py`, `Flask.create_url_adapter` binds the application's URL map (`url_map`) to the request environment using `bind_to_environ` (or `bind` if outside a request context):

```python
def create_url_adapter(self, request: Request | None) -> MapAdapter | None:
    ...
    return self.url_map.bind_to_environ(
        request.environ, server_name=server_name, subdomain=subdomain
    )
```

### 2. Request Matching
In `src/

## Step 10: Chat interface with memory (streaming)

In [35]:
class RAGChat:
    def __init__(self, top_k=5):
        self.top_k = top_k
        self.history = []  # list of (role, text) tuples

    def send(self, message, verbose=False):
        retrieved = retrieve(message, top_k=self.top_k)
        context_block = "\n\n".join(
            f"[{r['file']} :: {r['name']}]\n{r['text']}" for r in retrieved
        )

        history_block = "\n".join(f"{role}: {text}" for role, text in self.history[-6:])

        full_prompt = f"""{RAG_SYSTEM_PROMPT}

Conversation so far:
{history_block}

Context:
{context_block}

Question: {message}

Answer:"""

        if verbose:
            print("--- Retrieved chunks ---")
            for r in retrieved:
                print(f"[{r['file']} :: {r['name']}] distance={r['distance']:.3f}")
            print()

        print("--- streaming ---")
        full_text = ""
        for chunk in stream_with_retry(model=CHAT_MODEL, contents=full_prompt):
            if chunk.text:
                print(chunk.text, end="", flush=True)
                full_text += chunk.text
        print()

        cleaned = clean_response(full_text)
        print("\n--- clean version ---")
        print(cleaned)

        self.history.append(("User", message))
        self.history.append(("Assistant", cleaned))
        return cleaned

    def reset(self):
        self.history = []


chat = RAGChat(top_k=5)
chat.send("What does the Flask class do, at a high level?")

Embedded 1/1 chunks...
--- streaming ---
Based on the docstring in `src/flask/app.py`, `Flask`:

1. **Implements a WSGI application**: It serves as the main WSGI application object called by WSGI servers.
2. **Acts as a central registry**: Once created, it records and manages view functions, URL rules, template configurations, and other application-level settings.
3. **Resolves resources**: Takes an application module or package name (`import_name`) upon initialization to locate resources on the filesystem (such as templates and static files).

--- clean version ---
Based on the docstring in `src/flask/app.py`, `Flask`:
Implements a WSGI application: It serves as the main WSGI application object called by WSGI servers.
Acts as a central registry: Once created, it records and manages view functions, URL rules, template configurations, and other application-level settings.
Resolves resources: Takes an application module or package name (`import_name`) upon initialization to locate resour

'Based on the docstring in `src/flask/app.py`, `Flask`:\nImplements a WSGI application: It serves as the main WSGI application object called by WSGI servers.\nActs as a central registry: Once created, it records and manages view functions, URL rules, template configurations, and other application-level settings.\nResolves resources: Takes an application module or package name (`import_name`) upon initialization to locate resources on the filesystem (such as templates and static files).'

In [36]:
# Follow-up - the class remembers the previous turn
chat.send("Which method would I use to register a new route?")

Embedded 1/1 chunks...
--- streaming ---
To register a new route, you can use:

1. **`Scaffold.route`** (in `src/flask/sansio/scaffold.py`): A decorator used to register a view function for a given URL rule.
2. **`App.add_url_rule` / `Scaffold.add_url_rule`** (in `src/flask/sansio/app.py` and `src/flask/sansio/scaffold.py`): The underlying method that registers the URL rule, endpoint, and view function. `Scaffold.route` calls this method.
3. **HTTP method shortcuts** like `Scaffold.get` or `Scaffold.post` (in `src/flask/sansio/scaffold.py`): Decorator shortcuts for `route` with pre-set HTTP methods.

### Example usage:

Using the `route` decorator:
```python
@app.route("/")
def index():
    return "Hello, World!"
```

Using `add_url_rule`:
```python
def index():
    return "Hello, World!"

app.add_url_rule("/", view_func=index)
```

--- clean version ---
To register a new route, you can use:
`Scaffold.route` (in `src/flask/sansio/scaffold.py`): A decorator used to register a view funct

'To register a new route, you can use:\n`Scaffold.route` (in `src/flask/sansio/scaffold.py`): A decorator used to register a view function for a given URL rule.\n`App.add_url_rule` / `Scaffold.add_url_rule` (in `src/flask/sansio/app.py` and `src/flask/sansio/scaffold.py`): The underlying method that registers the URL rule, endpoint, and view function. `Scaffold.route` calls this method.\nHTTP method shortcuts like `Scaffold.get` or `Scaffold.post` (in `src/flask/sansio/scaffold.py`): Decorator shortcuts for `route` with pre-set HTTP methods.\n\nExample usage:\n\nUsing the `route` decorator:\n```python\n@app.route("/")\ndef index():\n    return "Hello, World!"\n```\n\nUsing `add_url_rule`:\n```python\ndef index():\n    return "Hello, World!"\n\napp.add_url_rule("/", view_func=index)\n```'

## Step 11: Evaluation framework

Adapted for code Q&A: does retrieval surface the right file/function, and does grounding actually reduce hallucination vs. asking Gemini with no code context?

In [37]:
# Replace these with real questions about Flask's actual source.
# expected_source should match a 'file' value from your chunks (e.g. "src/flask/app.py").
test_set = [
    {
        "question": "How does Flask register a URL route?",
        "expected_source": "src/flask/sansio/scaffold.py",
        "expected_answer_contains": "add_url_rule",
    },
    {
        "question": "How does Flask render templates?",
        "expected_source": "src/flask/templating.py",
        "expected_answer_contains": "render_template",
    },
    # add more real cases - check actual file paths after Step 4 runs, they may shift between Flask versions
]

print(f"Test set size: {len(test_set)}")
print("Note: verify 'expected_source' paths match what Step 4 actually found (Flask's internal layout can change between versions).")

Test set size: 2
Note: verify 'expected_source' paths match what Step 4 actually found (Flask's internal layout can change between versions).


In [38]:
def evaluate_retrieval(test_set, top_k=5):
    hits = 0
    results = []
    for case in test_set:
        retrieved = retrieve(case["question"], top_k=top_k)
        retrieved_files = [r["file"] for r in retrieved]
        hit = case["expected_source"] in retrieved_files
        hits += hit
        results.append({"question": case["question"], "hit": hit, "retrieved_files": retrieved_files})

    accuracy = hits / len(test_set) if test_set else 0
    print(f"Retrieval accuracy (expected file in top-{top_k}): {accuracy:.1%}")
    return results

retrieval_results = evaluate_retrieval(test_set)

Embedded 1/1 chunks...
Embedded 1/1 chunks...
Retrieval accuracy (expected file in top-5): 100.0%


In [39]:
def compare_rag_vs_no_rag(test_set, top_k=5):
    """Compares grounded (RAG) answers against raw Gemini knowledge with no code context -
    useful for showing measurable hallucination reduction in your writeup. Prints retrieved
    chunks for each question so retrieval misses are diagnosable, not just answer failures."""
    comparison = []

    for case in test_set:
        retrieved = retrieve(case["question"], top_k=top_k)
        print(f"Q: {case['question']}")
        print("  Retrieved:", [f"{r['file']}::{r['name']}" for r in retrieved])

        rag_answer = rag_generate_stream(case["question"], top_k=top_k, verbose=False)

        no_rag_response = client.models.generate_content(
            model=CHAT_MODEL,
            contents=f"Answer this question about the Flask framework's internals: {case['question']}"
        )
        no_rag_answer = clean_response(no_rag_response.text)

        comparison.append({
            "question": case["question"],
            "retrieved": [r["file"] for r in retrieved],
            "rag_answer": rag_answer,
            "no_rag_answer": no_rag_answer,
            "expected_contains": case["expected_answer_contains"],
            "rag_contains_expected": case["expected_answer_contains"].lower() in rag_answer.lower(),
            "no_rag_contains_expected": case["expected_answer_contains"].lower() in no_rag_answer.lower(),
        })

    return comparison

comparison_results = compare_rag_vs_no_rag(test_set)

print("\n\n=== SUMMARY ===")
for r in comparison_results:
    print(f"Q: {r['question']}")
    print(f"  Retrieved files: {r['retrieved']}")
    print(f"  RAG contains expected fact: {r['rag_contains_expected']}")
    print(f"  No-RAG contains expected fact: {r['no_rag_contains_expected']}")

Embedded 1/1 chunks...
Q: How does Flask register a URL route?
  Retrieved: ['src/flask/sansio/scaffold.py::Scaffold.add_url_rule', 'src/flask/sansio/scaffold.py::Scaffold.route', 'src/flask/ctx.py::AppContext.match_request', 'src/flask/sansio/app.py::App.add_url_rule', 'src/flask/app.py::Flask.dispatch_request']
Embedded 1/1 chunks...
--- streaming ---
In Flask, URL routes are registered either using the `@app.route` decorator or by calling `add_url_rule` directly.

### 1. Route Decorator
In `src/flask/sansio/scaffold.py`, `Scaffold.route` decorates a view function and delegates route creation to `add_url_rule`:
```python
def decorator(f: T_route) -> T_route:
    endpoint = options.pop("endpoint", None)
    self.add_url_rule(rule, endpoint, f, **options)
    return f
```

### 2. URL Rule Registration Implementation
In `src/flask/sansio/app.py`, `App.add_url_rule` performs the route registration as follows:

1. **Endpoint Resolution**: If `endpoint` is not passed, it defaults to the en

## Notes for your writeup

- **AST-based chunking** vs. fixed-size chunking is a good experimental comparison to include — you could re-run the pipeline with simple character-based chunking (from the earlier document version of this notebook) and show retrieval accuracy differs, since code split mid-function tends to retrieve worse.
- **`gemini-embedding-001`** doesn't know it's embedding code specifically — it treats it as text. Some published work shows code-specialized embedding models outperform general text embedders on code retrieval; noting this as a limitation (and possible future work) strengthens a thesis discussion section.
- **Nested/inner functions are not split separately** in this chunker (see Step 5) — a deliberate granularity choice to avoid over-fragmenting closures. Worth stating explicitly as a design decision.
- **Flask's internal file layout can change between versions** — the `expected_source` paths in the test set were written against the current `main` branch at the time of writing; if you pin to a specific release tag instead of `main` (change `REPO_ZIP_URL` to e.g. `.../archive/refs/tags/3.1.0.zip`), your results become reproducible against a fixed commit, which matters for thesis reproducibility.
- **Streaming**: this notebook streams every generation call (`generate_content_stream`), matching how a real user-facing chat experience would feel. If you want faster batch evaluation runs (Step 11) without the token-by-token print overhead, swap those specific calls to non-streaming `generate_content` — streaming vs. non-streaming doesn't change output quality, only display behavior.
- **Free tier rate limits** apply to both embedding and chat calls — indexing the full Flask repo may take a few minutes and multiple embedding batches; if you hit `429` errors, add a short `time.sleep()` between batches in Step 6.